In [ ]:
from IPython.display import HTML, display
display(HTML('''
<div style="text-align:right;padding:8px 0;">
<button id="toggle-code-btn" style="
  padding:8px 18px;font-size:14px;cursor:pointer;
  background:#4a90d9;color:white;border:none;border-radius:5px;
  box-shadow:0 2px 4px rgba(0,0,0,0.2);
" onclick="
  var cells = document.querySelectorAll('.jp-Cell-inputWrapper');
  var btn = document.getElementById('toggle-code-btn');
  var show = btn.dataset.show !== 'true';
  btn.dataset.show = show;
  btn.textContent = show ? '\u9690\u85cf\u4ee3\u7801 \ud83d\udd12' : '\u663e\u793a\u4ee3\u7801 \ud83d\udd13';
  cells.forEach(function(c){ c.style.display = show ? '' : 'none'; });
">\u663e\u793a\u4ee3\u7801 \ud83d\udd13</button>
</div>
'''))


# 02 - 分辨率与文件大小：每张图片要花多少存储？

**Cambridge International AS & A Level Computer Science (9618) - Section 1.2**

---

> "一张 4K 照片有 800 多万个像素，每个像素 3 个字节——就这么简单的乘法，一张照片就吃掉了 25MB。"

在这一节中：
- 什么是图像分辨率 vs 屏幕分辨率？
- 文件大小的计算公式（考试必考！）
- 像素密度 PPI 是什么？
- 为什么你买的 "1TB" 硬盘实际没有 1TB？

In [ ]:
%matplotlib inline

# === 中文字体配置 (Chinese Font Setup) ===
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='matplotlib')

# 方法1: 全局设置 WenQuanYi Zen Hei 字体
plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'Noto Sans CJK SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 方法2: 强制重建字体缓存（首次运行可能需要）
fm._load_fontmanager(try_read_cache=False)

# 验证
test_font = fm.findfont('WenQuanYi Zen Hei')
if 'WenQuanYi Zen Hei' in test_font or 'wqy' in test_font.lower():
    print('✅ 中文字体 WenQuanYi Zen Hei 已加载')
else:
    print(f'⚠️ WenQuanYi Zen Hei 未找到，使用: {test_font}')
    # Fallback: 直接注册字体文件
    font_path = '/usr/share/fonts/truetype/wqy/wqy-zenhei.ttc'
    if __import__('os').path.exists(font_path):
        fm.fontManager.addfont(font_path)
        plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei'] + plt.rcParams['font.sans-serif']
        print('✅ 已手动加载 WenQuanYi Zen Hei 字体文件')

import sys
try:
    import matplotlib.patches as patches
    from matplotlib.colors import ListedColormap
    import numpy as np
    from IPython.display import display, HTML, Image
    print('Libraries loaded!')
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'matplotlib', 'numpy', '-q'])
    import matplotlib.patches as patches
    from matplotlib.colors import ListedColormap
    import numpy as np
    from IPython.display import display, HTML, Image
    print('Installed & loaded!')
plt.rcParams.update({'font.size': 12, 'figure.figsize': (10,6),
    'font.sans-serif': ['WenQuanYi Zen Hei','Noto Sans CJK SC','DejaVu Sans'], 'axes.unicode_minus': False})


---
## 1. 图像分辨率 vs 屏幕分辨率

这两个概念容易混淆，但其实很不同：

### 1.1 图像分辨率 (Image Resolution)

指图像本身包含的 **总像素数** = 宽度像素 x 高度像素

例如：一张数码照片是 4096 x 3192 像素 = **13,074,432 像素 (约 1300 万)**

这就是相机标的 "1300 万像素" 的意思。

### 1.2 屏幕分辨率 (Screen Resolution)

指屏幕能显示的 **最大像素数**。

常见的屏幕分辨率：

| 名称 | 分辨率 | 总像素数 | 常见设备 |
|:---|:---:|---:|:---|
| HD (720p) | 1280 x 720 | 921,600 | 旧手机 |
| Full HD (1080p) | 1920 x 1080 | 2,073,600 | 主流笔记本 |
| 2K (QHD) | 2560 x 1440 | 3,686,400 | 高端手机/显示器 |
| 4K (UHD) | 3840 x 2160 | 8,294,400 | 高端电视/显示器 |
| 8K | 7680 x 4320 | 33,177,600 | 旗舰电视 |

### 1.3 当图像比屏幕大时会怎样？

如果你用 1920x1080 的屏幕显示 4096x3192 的照片：
- **整图显示**：软件必须缩小图片（丢弃像素），画质降低
- **100%显示**：只能看到图片的一部分，需要滚动查看
- **裁剪 (Crop)**：只保留一部分区域

这就是为什么 **图像分辨率不等于你看到的画质** ——还取决于屏幕能不能显示出来。

In [ ]:
# 可视化：图像分辨率 vs 屏幕分辨率
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Image smaller than screen
ax = axes[0]
screen = patches.Rectangle((0,0), 10, 6, fill=False, edgecolor='black', linewidth=3, linestyle='--')
image = patches.Rectangle((2, 1), 6, 4, facecolor='#90CAF9', edgecolor='blue', linewidth=2)
ax.add_patch(screen); ax.add_patch(image)
ax.text(5, -0.8, 'Screen 1920x1080', ha='center', fontsize=11)
ax.text(5, 3, 'Image\n1024x768', ha='center', va='center', fontsize=12, fontweight='bold')
ax.set_xlim(-1, 11); ax.set_ylim(-1.5, 7)
ax.set_title('Image < Screen\nFits with room to spare', fontsize=12, fontweight='bold', color='green')
ax.set_aspect('equal'); ax.axis('off')

# 2. Image equals screen
ax = axes[1]
screen = patches.Rectangle((0,0), 10, 6, fill=False, edgecolor='black', linewidth=3, linestyle='--')
image = patches.Rectangle((0, 0), 10, 6, facecolor='#A5D6A7', edgecolor='green', linewidth=2)
ax.add_patch(screen); ax.add_patch(image)
ax.text(5, -0.8, 'Screen 1920x1080', ha='center', fontsize=11)
ax.text(5, 3, 'Image\n1920x1080', ha='center', va='center', fontsize=12, fontweight='bold')
ax.set_xlim(-1, 11); ax.set_ylim(-1.5, 7)
ax.set_title('Image = Screen\nPerfect fit!', fontsize=12, fontweight='bold', color='green')
ax.set_aspect('equal'); ax.axis('off')

# 3. Image larger than screen
ax = axes[2]
screen = patches.Rectangle((2,1), 6, 4, fill=False, edgecolor='black', linewidth=3, linestyle='--')
image = patches.Rectangle((0, 0), 10, 6, facecolor='#FFCDD2', edgecolor='red', linewidth=2)
ax.add_patch(image); ax.add_patch(screen)
ax.text(5, -0.8, 'Screen 1920x1080', ha='center', fontsize=11)
ax.text(5, 3, 'Image 4096x3192\n(much bigger!)', ha='center', va='center', fontsize=12, fontweight='bold')
ax.set_xlim(-1, 11); ax.set_ylim(-1.5, 7)
ax.set_title('Image > Screen\nMust resize or crop!', fontsize=12, fontweight='bold', color='red')
ax.set_aspect('equal'); ax.axis('off')

plt.suptitle('Image Resolution vs Screen Resolution', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 2. 文件大小计算（考试重点！）

### 2.1 核心公式

$$\boxed{\text{文件大小 (bits)} = \text{宽度 (px)} \times \text{高度 (px)} \times \text{色彩深度 (bits)}}$$

$$\text{文件大小 (bytes)} = \frac{\text{文件大小 (bits)}}{8}$$

**为什么除以 8？**

因为 1 byte = 8 bits。回想第一章：byte 是计算机存储的基本单位。

### 2.2 单位换算

| 从 | 到 | 操作 |
|:---|:---|:---|
| bits | bytes | / 8 |
| bytes | KB | / 1,000 (SI) 或 / 1,024 (IEC) |
| KB | MB | / 1,000 (SI) 或 / 1,024 (IEC) |
| MB | GB | / 1,000 (SI) 或 / 1,024 (IEC) |

> 提醒：考试中要注意题目用的是 SI 还是 IEC 单位！

### 2.3 详细计算示例

**例题：** Full HD 屏幕截图，24 位色彩

```
步骤 1: 总像素数 = 1920 x 1080 = 2,073,600 pixels
步骤 2: 总位数   = 2,073,600 x 24 = 49,766,400 bits
步骤 3: 总字节   = 49,766,400 / 8 = 6,220,800 bytes
步骤 4: 转 MB    = 6,220,800 / 1,000,000 = 6.22 MB (SI)
                 = 6,220,800 / 1,048,576 = 5.93 MiB (IEC)
```

**注意：** 这是未压缩的 raw 大小。实际的 JPEG/PNG 文件会小得多（压缩的效果）。

In [ ]:
# 详细文件大小计算器
def calc_image_size(width, height, depth, show_steps=True):
    total_px = width * height
    total_bits = total_px * depth
    total_bytes = total_bits / 8
    kb_si = total_bytes / 1000
    mb_si = total_bytes / 1_000_000
    mb_iec = total_bytes / (1024 * 1024)

    if show_steps:
        print('=' * 60)
        print(f'  Image File Size Calculator')
        print('=' * 60)
        print(f'  Input:')
        print(f'    Width:        {width:,} pixels')
        print(f'    Height:       {height:,} pixels')
        print(f'    Colour depth: {depth} bits ({2**depth:,} colours)')
        print()
        print(f'  Step 1: Total pixels = {width:,} x {height:,} = {total_px:,}')
        print(f'  Step 2: Total bits   = {total_px:,} x {depth} = {total_bits:,} bits')
        print(f'  Step 3: Total bytes  = {total_bits:,} / 8 = {total_bytes:,.0f} bytes')
        print(f'  Step 4: Convert:')
        print(f'    = {kb_si:,.1f} KB  (SI, /1000)')
        print(f'    = {mb_si:,.2f} MB  (SI, /1000000)')
        print(f'    = {mb_iec:,.2f} MiB (IEC, /1048576)')
        print('=' * 60)
    return total_bytes

print('Example 1: Full HD screenshot (24-bit)')
calc_image_size(1920, 1080, 24)

print('\nExample 2: 4K photo (24-bit)')
calc_image_size(3840, 2160, 24)

print('\nExample 3: Tiny icon 32x32 (8-bit)')
calc_image_size(32, 32, 8)

print('\nExample 4: Black & white fax (1-bit)')
calc_image_size(1728, 1200, 1)

In [ ]:
# Try your own values!
my_width = 1024
my_height = 768
my_depth = 16

calc_image_size(my_width, my_height, my_depth)

In [ ]:
# Visual comparison: resolution vs file size
import math

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Chart 1: Resolution comparison
resolutions = {
    'Icon\n32x32': (32, 32),
    'VGA\n640x480': (640, 480),
    'HD\n1280x720': (1280, 720),
    'FHD\n1920x1080': (1920, 1080),
    '4K\n3840x2160': (3840, 2160),
    '8K\n7680x4320': (7680, 4320),
}
names = list(resolutions.keys())
sizes_mb = [(w * h * 24) / 8 / 1e6 for w, h in resolutions.values()]
colours = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(names)))
bars = ax1.bar(names, sizes_mb, color=colours, edgecolor='black', linewidth=1)
for bar, s in zip(bars, sizes_mb):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
             f'{s:.1f}MB', ha='center', fontsize=9, fontweight='bold')
ax1.set_ylabel('File Size (MB)')
ax1.set_title('File Size by Resolution\n(24-bit colour)', fontsize=14, fontweight='bold')
ax1.spines['top'].set_visible(False); ax1.spines['right'].set_visible(False)

# Chart 2: Colour depth comparison (at FHD)
depths = [1, 4, 8, 16, 24, 32]
sizes_depth = [(1920*1080*d)/8/1e6 for d in depths]
bars2 = ax2.bar([f'{d}-bit' for d in depths], sizes_depth,
                color=plt.cm.cool(np.linspace(0, 1, len(depths))),
                edgecolor='black', linewidth=1)
for bar, s in zip(bars2, sizes_depth):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2,
             f'{s:.1f}MB', ha='center', fontsize=9, fontweight='bold')
ax2.set_ylabel('File Size (MB)')
ax2.set_title('File Size by Colour Depth\n(FHD 1920x1080)', fontsize=14, fontweight='bold')
ax2.spines['top'].set_visible(False); ax2.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

print('Both resolution AND colour depth affect file size.')
print('Doubling resolution -> 4x file size (width x2 AND height x2)')
print('Doubling colour depth -> 2x file size')

---
## 3. 像素密度 PPI

### 3.1 什么是 PPI？

**PPI = Pixels Per Inch**（每英寸像素数）

同样分辨率的屏幕，尺寸越小，PPI 越高，画面越清晰。

### 3.2 计算公式

$$PPI = \frac{\sqrt{W^2 + H^2}}{D}$$

其中：
- W = 水平像素数
- H = 垂直像素数
- D = 屏幕对角线长度（英寸）
- $\sqrt{W^2 + H^2}$ = 对角线上的像素数（勾股定理！）

### 3.3 为什么用对角线？

因为屏幕尺寸（如 "5.5 英寸"）是按对角线量的，所以我们也要算对角线上的像素数，两者相除才有意义。

In [ ]:
# PPI Calculator with visual
import math

def calc_ppi(w, h, inches, name=''):
    diag_px = math.sqrt(w**2 + h**2)
    ppi = diag_px / inches
    print(f'  {name:22s} | {w}x{h:>4} | {inches:>5.1f}" | diag={diag_px:,.0f}px | PPI = {ppi:.0f}')
    return ppi

print(f'  {"Device":22s} | {"Resolution":>9} | {"Size":>5} | {"Diagonal":>10} | PPI')
print('  ' + '-' * 75)

devices = [
    (2556, 1179, 6.1, 'iPhone 15'),
    (1920, 1080, 6.5, 'Budget Android'),
    (2560, 1600, 13.3, 'MacBook Air 13"'),
    (1920, 1080, 15.6, 'Laptop 15.6"'),
    (3840, 2160, 27, '4K Monitor 27"'),
    (1920, 1080, 55, 'TV 55"'),
    (3840, 2160, 65, '4K TV 65"'),
]

ppis = []
for w, h, s, n in devices:
    ppis.append(calc_ppi(w, h, s, n))

print()
print('iPhone 15:  460 PPI - incredibly sharp, cannot see pixels')
print('TV 55":      40 PPI - much lower, but you sit far away')
print()
print('Key insight: PPI matters relative to viewing distance!')
print('Phones need high PPI (close viewing), TVs need less (far viewing).')

---
## 4. 文件头 (File Header)

实际的图像文件不只是像素数据。还包含一个 **文件头 (Header)**：

| 信息 | 说明 |
|:---|:---|
| 文件类型 | .bmp, .jpeg, .png 等 |
| 图像尺寸 | 宽度和高度（像素） |
| 色彩深度 | 每像素位数 |
| 压缩方式 | 是否压缩、用什么算法 |
| 其他元数据 | 拍摄时间、相机型号等 |

所以 **实际文件大小 = 像素数据 + 文件头**

文件头通常很小（几百字节），考试中一般忽略不计。

---
## 5. 练习题

**计算题：**
1. 一张 800x600 的图片，16 位色彩深度。计算文件大小（bytes、KB、MB）。
2. 4K 显示器 (3840x2160)，32 位色彩。一帧图像多少 MB？
3. 相机拍照 4096x3192，24 位。16GB 内存卡能存多少张未压缩照片？
4. 计算 iPhone 15 (2556x1179, 6.1") 的 PPI。

**思考题：**
5. 为什么同一张照片，在手机上看比在电视上看更清晰？
6. 一个 1920x1080 的视频帧，如果色彩深度从 24 位降到 8 位，文件大小变为原来的几分之几？画质会怎样？
7. 为什么你买的 "1TB 硬盘" 在电脑里只显示约 931 GB？

In [ ]:
# Answers
print('Q1:'); calc_image_size(800, 600, 16)
print('\nQ2:'); calc_image_size(3840, 2160, 32)
print('\nQ3:')
photo = 4096 * 3192 * 24 / 8
card = 16 * 1e9
print(f'  Photo size: {photo/1e6:.1f} MB')
print(f'  Card size: {card/1e6:.0f} MB')
print(f'  Photos: {card/photo:.0f}')

print('\nQ4:')
calc_ppi(2556, 1179, 6.1, 'iPhone 15')

print("""
Q5: The phone has much higher PPI (pixels per inch) than a TV.
    Even though the TV has more total pixels, they are spread over
    a much larger area, so each pixel is bigger and more visible.
    The phone packs more pixels into a smaller space = sharper image.

Q6: 8/24 = 1/3 of the original size.
    But colour quality drops from 16.7M colours to only 256 colours.
    Smooth gradients would show visible "banding" (colour steps).

Q7: Hard drive makers use SI (1 TB = 1,000,000,000,000 bytes)
    but computers use IEC (1 TiB = 1,099,511,627,776 bytes).
    1,000,000,000,000 / 1,099,511,627,776 = 0.909 TiB = ~931 GiB
    You are not losing space - it is just two different counting systems!
    (This connects back to Chapter 1.1 on binary/decimal prefixes)""")

---
**Next:** [03 - 矢量图形](03_矢量图形.ipynb)